In [ ]:
import os
import hashlib
import requests
from xml.etree import ElementTree as ET
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model

load_dotenv()


# Cloud LLMs
LLM_MODEL = os.getenv("LLM_MODEL")
LLM_PROVIDER = os.getenv("LLM_PROVIDER")

if LLM_PROVIDER == "anthropic":
    assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing from .env"

# Custom utils
DB_DIR = "./chroma_db"
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL, LLM_PROVIDER = "llama3.2", "ollama"

ACT_URLS = [
    "https://www.legislation.gov.uk/ukpga/2006/26/data.xml",
    "https://www.legislation.gov.uk/ukpga/2000/37/data.xml",
    "https://www.legislation.gov.uk/ukpga/1981/69/data.xml",
]



In [109]:
NS = "{http://www.legislation.gov.uk/namespaces/legislation}"


def load(url):
    req = requests.get(url)
    if req.status_code != 200:
        return []

    root = ET.fromstring(req.content)
    docs = []

    for p1 in root.iter(f"{NS}P1"):
        section = "".join(p1.find(f"{NS}Pnumber").itertext()).strip()
        p1_uri = p1.get("DocumentURI")

        for p2 in p1.iter(f"{NS}P2"):
            if p2.get("DocumentURI") is None:   # quoted text from another Act
                continue
            sub = "".join(p2.find(f"{NS}Pnumber").itertext()).strip()
            uri = f"{p1_uri}/{sub}"             # constructed, not read: source data has errors
            content = " ".join(
                t.strip() for el in p2.find(f"{NS}P2para").iter()
                if el.tag != f"{NS}Pnumber"
                for t in [el.text, el.tail] if t and t.strip()
            )
            docs.append(Document(
                page_content=f"Section {section}({sub}): {content}",
                metadata={"source": url, "section": section, "subsection": sub, "uri": uri},
            ))

    return docs

In [110]:
def chunk_id(doc):
    return hashlib.md5(doc.metadata["uri"].encode()).hexdigest()

def get_store():
    return Chroma(
        collection_name="legislation",
        embedding_function=OllamaEmbeddings(model=EMBED_MODEL),
        persist_directory=DB_DIR,
    )

In [111]:
def index(urls=ACT_URLS, rebuild=True):
    store = get_store()
    if rebuild:
        store.delete_collection()
        store = get_store()

    docs = []
    for url in urls:
        docs += load(url)
        
    store.add_documents(docs, ids=[chunk_id(d) for d in docs])
    print(f"{len(docs)} subsections indexed, store holds {store._collection.count()}")
    return store

In [112]:
def retrieve(store, question, k=4):
    return store.similarity_search_with_score(question, k=k)


def show(hits):
    for doc, score in hits:
        print(f"[{score:.3f}] s.{doc.metadata['section']}({doc.metadata['subsection']})")
        print("   ", doc.page_content[:200], "\n")

In [113]:
llm = init_chat_model(LLM_MODEL, model_provider=LLM_PROVIDER)


def build_prompt(question, hits):
    ctx = "\n\n".join(d.page_content for d, _ in hits)
    return (f"Answer using only the context below. Cite the section you used. "
            f"If the context does not contain the answer, say so.\n\n"
            f"Context:\n{ctx}\n\nQuestion: {question}")


def ask(store, question, k=4, debug=False):
    hits = retrieve(store, question, k)
    if debug:
        show(hits)
    return llm.invoke(build_prompt(question, hits)).text

In [114]:
docs = []
for url in ACT_URLS:
    d = load(url)
    print(url.split("/ukpga/")[1].split("/data")[0], len(d))
    docs += d

assert docs, "loader returned nothing"
assert all(d.metadata["uri"] for d in docs), "missing URI"
assert len({d.metadata["uri"] for d in docs}) == len(docs), "duplicate URIs"
assert all(len(d.page_content.split(": ", 1)[1]) > 0 for d in docs), "empty content"
print(f"{len(docs)} docs, all valid")

2006/26 382
2000/37 697
1981/69 1283
2362 docs, all valid


In [115]:
root = ET.fromstring(requests.get("https://www.legislation.gov.uk/ukpga/2000/37/data.xml").content)
for p1 in root.iter(f"{NS}P1"):
    if "".join(p1.find(f"{NS}Pnumber").itertext()).strip() == "68":
        print("P1 URI:", p1.get("DocumentURI"))
        for p2 in p1.iter(f"{NS}P2"):
            print("  P2:", "".join(p2.find(f"{NS}Pnumber").itertext()).strip(), p2.get("DocumentURI"))
        break

P1 URI: http://www.legislation.gov.uk/ukpga/2000/37/section/68
  P2: 1 http://www.legislation.gov.uk/ukpga/2000/37/section/15/1
  P2: 2 http://www.legislation.gov.uk/ukpga/2000/37/section/68/2
  P2: 3 http://www.legislation.gov.uk/ukpga/2000/37/section/68/3
  P2: 4 http://www.legislation.gov.uk/ukpga/2000/37/section/68/4
  P2: 5 http://www.legislation.gov.uk/ukpga/2000/37/section/68/5
  P2: 6 http://www.legislation.gov.uk/ukpga/2000/37/section/68/6
